# Step 2 — Run Flint on a Kaggle GPU and expose an OpenAI-compatible API

This notebook mounts your reusable Flint model Dataset and your prebuilt CUDA-enabled `llama-cpp-bin` Dataset, starts `llama-server`, and opens a Cloudflare Quick Tunnel.

### Architecture

```text
Kaggle GPU session
  Flint-Q4_K_M.gguf
          ↕
  llama-server :8080  ← OpenAI-compatible /v1 API
          ↕
  cloudflared Quick Tunnel
          ↕
  https://<temporary-name>.trycloudflare.com
          ↕
  local chat UI / coding agent
```

### Prerequisites
- Accelerator: **GPU T4 x2** recommended; T4 x1 may require CPU/RAM offload and be much slower.
- Internet: **ON**.
- Attach the private `flint-gguf` dataset created by `1_setup_flint_dataset.ipynb`.
- Attach `llama-cpp-bin`, rebuilt from a `llama.cpp` revision that includes Flint architecture support. Flint support landed upstream in October 2026; an older build can fail with `unknown model architecture: 'flint'`.
- This model is **text-only**. Do not configure `--mmproj`.

**Security note:** a Quick Tunnel gives you a public URL. Set an API key in Cell 1 if you want requests to require a key, and don't share the URL/key publicly. Kaggle sessions are temporary; the URL changes when a new tunnel is created.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — Edit this cell only
# ══════════════════════════════════════════════════════════════════════════════

KAGGLE_USERNAME = ""  # e.g. "johndoe"

MODEL_DATASET = "flint-gguf"
MODEL_FILE = "Flint-Q4_K_M.gguf"
LLAMA_DATASET = "llama-cpp-bin"

PORT = 8080
CTX_SIZE = 16384       # Start conservatively. Increase only after testing VRAM.
N_GPU_LAYERS = 999     # Offload as many layers as possible.
PARALLEL = 1           # One active request; helps preserve context memory.

# Flint recommended sampling defaults from the model information:
TEMPERATURE = 1.0
TOP_P = 0.95
TOP_K = 20

# Optional authentication for the public API. Leave "" to disable API-key auth.
# If set, clients must send this key as an Authorization: Bearer <key> header.
API_KEY = ""

# ══════════════════════════════════════════════════════════════════════════════
import os
from pathlib import Path

if not KAGGLE_USERNAME:
    KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "")
if not KAGGLE_USERNAME:
    raise ValueError("Please set KAGGLE_USERNAME at the top of this cell.")

def find_dataset(slug):
    # Kaggle's mount layout can vary between /kaggle/input/<slug> and
    # /kaggle/input/datasets/<username>/<slug>, so check both.
    candidates = [
        Path(f"/kaggle/input/datasets/{KAGGLE_USERNAME}/{slug}"),
        Path(f"/kaggle/input/{slug}"),
    ]
    for p in candidates:
        if p.exists():
            return p
    # Last-resort discovery for attached datasets.
    root = Path("/kaggle/input")
    if root.exists():
        matches = [p for p in root.rglob(slug) if p.is_dir() and p.name == slug]
        if matches:
            return matches[0]
    raise FileNotFoundError(
        f"Dataset '{slug}' not found. Attach it in the notebook's Input panel. "
        f"Checked: {', '.join(map(str, candidates))}"
    )

MODEL_DIR = find_dataset(MODEL_DATASET)
LLAMA_BIN_DIR = find_dataset(LLAMA_DATASET)
MODEL_PATH = MODEL_DIR / MODEL_FILE
WORKING = Path("/kaggle/working")
LLAMA_SERVER = WORKING / "llama-server"

assert MODEL_PATH.exists(), f"Model file not found: {MODEL_PATH}"
assert (LLAMA_BIN_DIR / "llama-server").exists(), f"llama-server not found in {LLAMA_BIN_DIR}"

print("Configuration validated")
print("Model:", MODEL_PATH)
print("Binaries:", LLAMA_BIN_DIR)
print("Context:", CTX_SIZE)
print("API key authentication:", "enabled" if API_KEY else "disabled")


In [ ]:
# Cell 2 — Check GPU(s)
import subprocess

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
    capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError("No NVIDIA GPU detected. Set Accelerator to GPU T4 in notebook settings.")

total_mib = 0
print(f"{'GPU':<5} {'Name':<22} {'Total VRAM':>12} {'Free VRAM':>12}")
print("-" * 57)
for i, line in enumerate(result.stdout.strip().splitlines()):
    parts = [x.strip() for x in line.split(",")]
    name, total, free = parts
    total_mb = int(total.replace(" MiB", ""))
    free_mb = int(free.replace(" MiB", ""))
    total_mib += total_mb
    print(f"{i:<5} {name:<22} {total_mb/1024:>9.1f} GiB {free_mb/1024:>9.1f} GiB")
print("-" * 57)
print(f"Total VRAM across visible GPUs: {total_mib/1024:.1f} GiB")
print(f"Model file size: {MODEL_PATH.stat().st_size / (1024**3):.2f} GiB")
print("Note: weights are not the only VRAM use; KV cache and compute buffers need additional memory.")


In [ ]:
# Cell 3 — Copy binaries and shared libraries into writable /kaggle/working
import shutil, json, os, subprocess
from pathlib import Path

WORKING.mkdir(parents=True, exist_ok=True)

for name in ["llama-server", "llama-cli", "llama-mtmd-cli"]:
    src = LLAMA_BIN_DIR / name
    if src.exists():
        dst = WORKING / name
        shutil.copy2(src, dst)
        dst.chmod(0o755)

so_count = 0
for so_file in LLAMA_BIN_DIR.glob("*.so*"):
    shutil.copy2(so_file, WORKING / so_file.name)
    so_count += 1

ld_path = f"{WORKING}:{os.environ.get('LD_LIBRARY_PATH', '')}"
os.environ["LD_LIBRARY_PATH"] = ld_path

test = subprocess.run(
    [str(LLAMA_SERVER), "--version"],
    capture_output=True, text=True,
    env={**os.environ, "LD_LIBRARY_PATH": ld_path},
)
if test.returncode != 0:
    print(test.stdout)
    print(test.stderr[-4000:])
    raise RuntimeError("llama-server failed its --version check. Rebuild the llama-cpp-bin dataset.")

version_file = LLAMA_BIN_DIR / "version.json"
if version_file.exists():
    info = json.loads(version_file.read_text())
    print("llama.cpp binary version check passed.")
    print("Commit:", info.get("llama_cpp_commit", "unknown"), "Date:", info.get("commit_date", "unknown"))
    print("Version:", info.get("llama_server_ver", (test.stdout or test.stderr).strip().splitlines()[0]))
    print("CUDA target:", info.get("cuda_arch", "unknown"))
else:
    print("llama.cpp binary version check passed:", (test.stdout or test.stderr).strip().splitlines()[0])

print(f"Copied binaries and {so_count} shared libraries.")


In [ ]:
# Cell 4 — Verify that this llama.cpp build recognizes Flint
# This catches the common failure mode before starting the long model load.
import subprocess, os

env = {**os.environ, "LD_LIBRARY_PATH": f"{WORKING}:{os.environ.get('LD_LIBRARY_PATH', '')}"}
help_result = subprocess.run(
    [str(LLAMA_SERVER), "--help"],
    capture_output=True, text=True, env=env, timeout=30
)
help_text = (help_result.stdout or "") + "\n" + (help_result.stderr or "")
if help_result.returncode != 0:
    print(help_text[-3000:])
    raise RuntimeError("llama-server --help failed.")

print("Binary runs. Flint architecture support is definitively checked when the model is loaded.")
print("If startup logs say 'unknown model architecture: flint', rebuild llama.cpp from a commit containing Flint support.")


In [ ]:
# Cell 5 — Install Cloudflare Tunnel
%%bash
set -e
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
chmod +x /usr/local/bin/cloudflared
cloudflared --version


In [ ]:
# Cell 6 — Launch llama-server (Flint is text-only; no --mmproj)
import subprocess, time, requests, os, signal
from pathlib import Path

# Stop a prior server process if this cell is rerun.
subprocess.run(["pkill", "-x", "llama-server"], capture_output=True)
time.sleep(1)

env = {**os.environ, "LD_LIBRARY_PATH": f"{WORKING}:{os.environ.get('LD_LIBRARY_PATH', '')}"}

cmd = [
    str(LLAMA_SERVER),
    "--model", str(MODEL_PATH),
    "--alias", "flint",
    "--host", "0.0.0.0",
    "--port", str(PORT),
    "--n-gpu-layers", str(N_GPU_LAYERS),
    "--ctx-size", str(CTX_SIZE),
    "--parallel", str(PARALLEL),
    "--temp", str(TEMPERATURE),
    "--top-p", str(TOP_P),
    "--top-k", str(TOP_K),
    "--log-file", "/tmp/llama-server.log",
]
if API_KEY:
    cmd += ["--api-key", API_KEY]

print("Starting Flint server...")
print("Model:", MODEL_FILE)
print("Context:", CTX_SIZE)
print("GPU layers:", N_GPU_LAYERS)
print("API key auth:", "enabled" if API_KEY else "disabled")

server_proc = subprocess.Popen(
    cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=env
)

timeout_seconds = 900
interval = 10
elapsed = 0
server_ok = False
while elapsed < timeout_seconds:
    if server_proc.poll() is not None:
        break
    try:
        response = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=3)
        if response.status_code == 200 and response.json().get("status") == "ok":
            server_ok = True
            break
    except Exception:
        pass
    print(f"[{elapsed:>3}s] Model is loading...", flush=True)
    time.sleep(interval)
    elapsed += interval

if not server_ok:
    print("\nLast server log lines:")
    try:
        print("\n".join(Path("/tmp/llama-server.log").read_text().splitlines()[-50:]))
    except Exception as e:
        print("Could not read log:", e)
    if server_proc.poll() is not None:
        print("Server exited with code:", server_proc.returncode)
    raise RuntimeError("Flint did not become healthy. Inspect logs above.")

print(f"\nFlint is ready after approximately {elapsed} seconds.")
print(f"Local OpenAI-compatible endpoint: http://127.0.0.1:{PORT}/v1")


In [ ]:
# Cell 7 — Start Cloudflare Quick Tunnel and print the public endpoint
import subprocess, time, re
from pathlib import Path

tunnel_log_path = Path("/tmp/cloudflare.log")
tunnel_log_path.write_text("")

tunnel_log_handle = open(tunnel_log_path, "a")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=tunnel_log_handle,
    stderr=tunnel_log_handle,
)

tunnel_url = None
for _ in range(45):
    time.sleep(2)
    log_text = tunnel_log_path.read_text(errors="replace")
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_text)
    if match:
        tunnel_url = match.group(0)
        break
    if tunnel_proc.poll() is not None:
        break

if not tunnel_url:
    print(tunnel_log_path.read_text(errors="replace")[-3000:])
    raise RuntimeError("Could not obtain a Cloudflare Quick Tunnel URL.")

print("=" * 68)
print("FLINT SERVER URL")
print(tunnel_url)
print("=" * 68)
print("OpenAI-compatible base URL:", tunnel_url + "/v1")
print("Model alias: flint")
if API_KEY:
    print("Authentication is enabled. Configure clients to send your API key.")
else:
    print("WARNING: API key authentication is disabled; anyone with this URL may be able to use the API.")
print("The URL is temporary and will change on a new tunnel/session.")


In [ ]:
# Cell 8 — Quick API smoke test
# If API_KEY is set, send it as a Bearer token.
import requests

headers = {}
if API_KEY:
    headers["Authorization"] = f"Bearer {API_KEY}"

try:
    r = requests.get(f"http://127.0.0.1:{PORT}/v1/models", headers=headers, timeout=10)
    print("HTTP status:", r.status_code)
    print(r.text[:2000])
    r.raise_for_status()
except Exception as e:
    print("API smoke test failed:", repr(e))
    print("Check /tmp/llama-server.log and the authentication setting.")


In [ ]:
# Cell 9 — Keep the server alive and report health every 5 minutes
import datetime, time, requests

start_time = datetime.datetime.now()
check_interval = 300

print("Flint server started:", start_time.strftime("%Y-%m-%d %H:%M:%S"))
print("Public URL:", tunnel_url)
print("Base API URL:", tunnel_url + "/v1")
print("Model alias: flint")
print("Stop by interrupting this cell.")
print("-" * 68)

try:
    while True:
        time.sleep(check_interval)
        elapsed = datetime.datetime.now() - start_time
        h, rem = divmod(int(elapsed.total_seconds()), 3600)
        m = rem // 60
        try:
            r = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=5)
            status = "healthy" if r.status_code == 200 and r.json().get("status") == "ok" else f"HTTP {r.status_code}"
        except Exception:
            status = "unreachable"
        print(f"[{datetime.datetime.now():%H:%M:%S}] Uptime {h}h {m:02d}m | {status}", flush=True)
        if status == "unreachable":
            print("Server appears unreachable. Check /tmp/llama-server.log")
            break
except KeyboardInterrupt:
    print("\nStopping server and tunnel...")
    for proc in [server_proc, tunnel_proc]:
        try:
            proc.terminate()
        except Exception:
            pass
    try:
        tunnel_log_handle.close()
    except Exception:
        pass
    print("Stopped.")


## Notes

- **No `mmproj`:** Flint is text-only, unlike your previous Qwen3.6 notebook.
- **Context:** Flint's architecture advertises up to 262K tokens, but that does not mean a 2×T4 setup can run that context. Begin at 16K; if it loads reliably, test 32K and 64K while monitoring VRAM. Large contexts consume extra KV-cache memory.
- **If you see `unknown model architecture: 'flint'`:** the attached `llama-cpp-bin` is too old or was built from a commit without Flint support. Rebuild the binary dataset from a compatible current `llama.cpp` commit.
- **If it runs out of VRAM:** reduce `CTX_SIZE` first, or use a smaller quantization / partial CPU offload.
- **API clients:** use `https://<your-tunnel>.trycloudflare.com/v1` as the base URL and `flint` as the model name. If you set `API_KEY`, configure the client to send it as a Bearer token.
- **Security:** a Cloudflare Quick Tunnel URL is publicly reachable while active. Use authentication if possible and stop the notebook when you're done.
